## Random Forest

O [Random Forest](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html) é um modelo de ensemble que combina múltiplas árvores de decisão para melhorar a performance e reduzir o overfitting.

Cada árvore é treinada em uma amostra diferente dos dados, e a decisão final é feita por votação.

Esse modelo tende a ser mais robusto e apresentar melhor generalização em comparação com uma única árvore.

### Hiperparâmetros utilizados

- **n_estimators**: número de árvores na floresta.
  - Mais árvores → melhor desempenho (até certo ponto)

- **max_depth**: profundidade máxima das árvores.
  - Controla o overfitting

- **min_samples_split**: mínimo de amostras para dividir um nó

- **min_samples_leaf**: O número mínimo de amostras necessário para que um nó seja uma folha

### Estratégia

Foi utilizado GridSearchCV para encontrar a melhor combinação de hiperparâmetros, utilizando validação cruzada e a métrica ROC AUC.

In [3]:
import pandas as pd
import sys
import os

# add raiz do projeto
sys.path.append(os.path.abspath(".."))

from sklearn.model_selection import GridSearchCV
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score

from preprocessing.main_preprocessing import load_and_preprocess
from utils.experiment_logger import save_results


## BASELINE

In [4]:
scenarios = [
    "sem_submodalidade",
    "submodalidade_agrupada",
    "submodalidade_engineered"
]

smote_options = [False, True]

results = []


for scenario in scenarios:
    for use_smote in smote_options:

        print(f"\n🔎 Scenario: {scenario} | SMOTE: {use_smote}")

        X_train, X_test, y_train, y_test = load_and_preprocess(
            "../predictive_models/scrdata_202505.csv",
            scenario=scenario,
            use_smote=use_smote
        )

        steps = [
            ('rf', RandomForestClassifier(random_state=42,
            n_jobs=-1))
        ]

        if use_smote:
            steps.insert(1 if not steps or steps[0][0] != 'scaler' else 1, ('smote', SMOTE(random_state=42)))

        model = Pipeline(steps)

        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1]

        results.append({
            "model": "RandomForest",
            "scenario": scenario,
            "smote": use_smote,
            "roc_auc": roc_auc_score(y_test, y_proba),
            "f1": f1_score(y_test, y_pred),
            "accuracy": accuracy_score(y_test, y_pred),
            "n_features": X_train.shape[1],
            "phase": "baseline",
            "timestamp": pd.Timestamp.now()
        })

#save_results(results, file_path="../results/experiments.csv")

df_result = pd.DataFrame(results)

display(df_result)



🔎 Scenario: sem_submodalidade | SMOTE: False

🔎 Scenario: sem_submodalidade | SMOTE: True

🔎 Scenario: submodalidade_agrupada | SMOTE: False

🔎 Scenario: submodalidade_agrupada | SMOTE: True

🔎 Scenario: submodalidade_engineered | SMOTE: False

🔎 Scenario: submodalidade_engineered | SMOTE: True


,model,scenario,smote,roc_auc,f1,accuracy,n_features,phase,timestamp
0,RandomForest,sem_submodalidade,False,0.930511,0.871706,0.850830,67,baseline,2026-04-09 23:06:54.804912
1,RandomForest,sem_submodalidade,True,0.929446,0.867909,0.849078,67,baseline,2026-04-09 23:07:04.644533
2,RandomForest,submodalidade_agrupada,False,0.943238,0.886033,0.867129,97,baseline,2026-04-09 23:07:10.637283
3,RandomForest,submodalidade_agrupada,True,0.942463,0.882779,0.865158,97,baseline,2026-04-09 23:07:21.558167
4,RandomForest,submodalidade_engineered,False,0.930511,0.871706,0.850830,67,baseline,2026-04-09 23:07:27.210983
5,RandomForest,submodalidade_engineered,True,0.929446,0.867909,0.849078,67,baseline,2026-04-09 23:07:36.941425


## GRIDSEARCH

In [5]:
X_train, X_test, y_train, y_test = load_and_preprocess(
    "../predictive_models/scrdata_202505.csv",
    scenario="sem_submodalidade",
    use_smote=False
)


param_grid_rf = {
    "smote": [SMOTE(random_state=42), "passthrough"],
    "rf__n_estimators": [100, 200],
    "rf__min_samples_split": [2, 5],
    "rf__min_samples_leaf": [1, 2]
}

grid_rf = GridSearchCV(
    estimator=Pipeline([
        ('smote', SMOTE(random_state=42)),
        ('rf', RandomForestClassifier(random_state=42,
        n_jobs=-1))
    ]),
    param_grid=param_grid_rf,
    cv=5,
    scoring="roc_auc",
    n_jobs=1
)

grid_rf.fit(X_train, y_train)

print("Best parameters:", grid_rf.best_params_)
print("Best ROC AUC (CV):", grid_rf.best_score_)


#? BEST MODEL TEST EVALUATION

best_rf = grid_rf.best_estimator_

y_pred = best_rf.predict(X_test)
y_proba = best_rf.predict_proba(X_test)[:, 1]


#? TUNING (CV)

tuning_result = [{
    "model": "RandomForest",
    "scenario": "sem_submodalidade",
    "smote": False,
    "phase": "tuning_cv",
    "roc_auc": grid_rf.best_score_,
    "f1": None,
    "accuracy": None,
    "best_params": str(grid_rf.best_params_),
    "timestamp": pd.Timestamp.now()
}]

final_result = [{
    "model": "RandomForest",
    "scenario": "sem_submodalidade",
    "smote": False,
    "phase": "test",
    "roc_auc": roc_auc_score(y_test, y_proba),
    "f1": f1_score(y_test, y_pred),
    "accuracy": accuracy_score(y_test, y_pred),
    "best_params": str(grid_rf.best_params_),
    "timestamp": pd.Timestamp.now()
}]


save_results(tuning_result, file_path="../results/model_results.csv")
save_results(final_result, file_path="../results/model_results.csv")


df_result = pd.DataFrame(final_result)
display(df_result)


Best parameters: {'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 200}
Best ROC AUC (CV): 0.9321227897368385


,model,scenario,smote,phase,roc_auc,f1,accuracy,best_params,timestamp
0,RandomForest,sem_submodalidade,False,test,0.93219,0.874057,0.853732,"{'min_samples_leaf': 2, 'min_samples_split': 2...",2026-04-09 23:10:35.375419
